# AI Model Validation: From Mathematics to Trust
## Episode 04 — From Error to Evidence: MAE, MSE, RMSE and the Mathematics of Model Error

**Author: Vishwanath Akuthota**  
**Founder of DrPinnacle, OpenVals and The Foundry's**

An AI model can be wrong by 1 unit, 10 units, or 1,000 units. How should those errors be combined into scientifically meaningful evidence?

> **There is no universally best error metric. The metric defines what kind of error the evaluation considers important.**

This episode develops residual error, MAE, MSE, RMSE, bias, outliers, scale, L1/L2 loss, tail error, baselines and sampling uncertainty.


## 1. Prediction Error

For observation *i*, define the signed residual:

### 𝑒ᵢ = 𝑦ᵢ − ŷᵢ

Absolute error:

### |𝑒ᵢ| = |𝑦ᵢ − ŷᵢ|

Squared error:

### 𝑒ᵢ² = (𝑦ᵢ − ŷᵢ)²

A signed residual preserves direction. Absolute and squared errors measure magnitude in different ways.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import platform, sys

print("Python :", sys.version.split()[0])
print("System :", platform.system(), platform.release())
print("Machine:", platform.machine())
print("NumPy  :", np.__version__)

y_true=np.array([10.,20.,30.,40.,50.])
y_pred=np.array([12.,18.,29.,44.,45.])
residuals=y_true-y_pred

print("\n y     y_hat   residual   |error|   error^2")
print("-"*51)
for y,p,r in zip(y_true,y_pred,residuals):
    print(f"{y:>4.0f}   {p:>6.0f}   {r:>8.0f}   {abs(r):>7.0f}   {r**2:>7.0f}")


## 2. Mean Error — Directional Bias

### ME = (1/n) Σᵢ₌₁ⁿ (𝑦ᵢ − ŷᵢ)

Mean Error can reveal systematic overprediction or underprediction under this sign convention.

But positive and negative residuals can cancel. A Mean Error near zero therefore does **not** prove that predictions are accurate.


In [ ]:
print("Residuals :", residuals)
print("Mean Error:", np.mean(residuals))


## 3. Mean Absolute Error — MAE

### MAE = (1/n) Σᵢ₌₁ⁿ |𝑦ᵢ − ŷᵢ|

MAE measures average absolute prediction error and remains in the original target units.

An MAE of 3 means that predictions differ from observed targets by 3 target-units on average in absolute magnitude. It does not mean every prediction is wrong by exactly 3 units.


In [ ]:
mae=np.mean(np.abs(residuals))
print(f"MAE = {mae:.4f}")


## 4. Mean Squared Error — MSE

### MSE = (1/n) Σᵢ₌₁ⁿ (𝑦ᵢ − ŷᵢ)²

Squaring causes larger residuals to receive disproportionately greater numerical weight.

MSE is expressed in squared target units.


In [ ]:
mse=np.mean(residuals**2)
print(f"MSE = {mse:.4f}")


## 5. Root Mean Squared Error — RMSE

### RMSE = √[(1/n) Σᵢ₌₁ⁿ (𝑦ᵢ − ŷᵢ)²]

RMSE returns squared error to the original target units while retaining stronger sensitivity to large residuals.


In [ ]:
rmse=np.sqrt(mse)
print(f"RMSE = {rmse:.4f}")


## 6. MAE and RMSE Are Not Interchangeable

For a finite collection of residuals:

### MAE ≤ RMSE

Equality occurs when all absolute residuals are equal. The gap grows as residual magnitudes become more uneven.


In [ ]:
examples={
    "Equal errors":np.array([2.,-2.,2.,-2.]),
    "Uneven errors":np.array([0.,0.,0.,8.]),
    "Mixed errors":np.array([1.,-2.,3.,-6.])
}
for name,r in examples.items():
    mae=np.mean(np.abs(r))
    rmse=np.sqrt(np.mean(r**2))
    print(f"{name:<15} MAE={mae:.3f} RMSE={rmse:.3f} gap={rmse-mae:.3f}")


## 7. Why Squaring Changes the Evaluation

Absolute loss:

### L₁(e) = |e|

Squared loss:

### L₂(e) = e²

A residual twice as large produces twice the absolute loss but four times the squared loss.


In [ ]:
err=np.linspace(-10,10,500)
plt.figure(figsize=(8,5))
plt.plot(err,np.abs(err),label="Absolute loss |e|")
plt.plot(err,err**2,label="Squared loss e²")
plt.xlabel("Residual e"); plt.ylabel("Loss")
plt.title("Absolute Loss vs Squared Loss")
plt.legend(); plt.grid(alpha=.25); plt.show()


## 8. Experiment — One Outlier Changes the Story

Large residuals influence all error metrics, but squared-error metrics react much more strongly.

That is neither automatically good nor bad. It depends on whether large errors are disproportionately costly in the intended application.


In [ ]:
normal=np.array([1.,-1.,2.,-2.,1.,-1.,2.,-2.])
outlier=np.append(normal,20.)

def summarize(r):
    return np.mean(np.abs(r)),np.mean(r**2),np.sqrt(np.mean(r**2))

for label,r in [("Without outlier",normal),("With outlier",outlier)]:
    a,b,d=summarize(r)
    print(f"{label:<16} MAE={a:>6.3f} MSE={b:>8.3f} RMSE={d:>6.3f}")


## 9. Error Metrics Encode Consequences

For errors of 5 and 10:

### |10| / |5| = 2

but:

### 10² / 5² = 4

The mathematical choice embeds an implicit statement about how rapidly the penalty grows with error magnitude.


## 10. L1 and L2 Loss

MAE is associated with absolute or L1 loss:

### L₁ = Σᵢ₌₁ⁿ |𝑦ᵢ − ŷᵢ|

Squared-error optimization uses:

### L₂² = Σᵢ₌₁ⁿ (𝑦ᵢ − ŷᵢ)²

These objectives have different mathematical and statistical properties and can favor different predictions.


## 11. Mean vs Median — A Fundamental Connection

For a constant prediction *c*, expected squared error is minimized by the mean under standard conditions:

### c*MSE = E[Y]

Expected absolute error is minimized by a median:

### c*MAE = Median(Y)

This is one reason MAE and MSE behave differently with skewed distributions and extreme observations.


In [ ]:
values=np.array([10.,11.,11.,12.,12.,13.,100.])
print("Values :",values)
print("Mean   :",values.mean())
print("Median :",np.median(values))

grid=np.linspace(5,105,600)
mae_loss=np.array([np.mean(np.abs(values-g)) for g in grid])
mse_loss=np.array([np.mean((values-g)**2) for g in grid])

plt.figure(figsize=(8,5))
plt.plot(grid,mae_loss)
plt.axvline(np.median(values),linestyle="--",label="Median")
plt.xlabel("Constant prediction"); plt.ylabel("MAE loss")
plt.title("Absolute Error Is Minimized at a Median")
plt.legend(); plt.grid(alpha=.25); plt.show()

plt.figure(figsize=(8,5))
plt.plot(grid,mse_loss)
plt.axvline(values.mean(),linestyle="--",label="Mean")
plt.xlabel("Constant prediction"); plt.ylabel("MSE loss")
plt.title("Squared Error Is Minimized at the Mean")
plt.legend(); plt.grid(alpha=.25); plt.show()


## 12. Differentiability and Optimization

For squared loss:

### L = (𝑦 − ŷ)²

### ∂L/∂ŷ = 2(ŷ − 𝑦)

Absolute loss, |𝑦 − ŷ|, has a nondifferentiable point at zero in the ordinary sense, although optimization can use subgradients and related methods.


## 13. Training Loss Is Not Complete Validation Evidence

### Training Objective ≠ Complete Validation Evidence

A model trained using MSE still needs evaluation for MAE, tail error, robustness, subgroup behaviour, safety, uncertainty, and the consequences relevant to its intended use.


## 14. Scale Matters

If both actual and predicted targets are multiplied by a constant *a*:

### 𝑦′ᵢ = a𝑦ᵢ ,  ŷ′ᵢ = aŷᵢ

then:

### MAE′ = |a| MAE

### RMSE′ = |a| RMSE

### MSE′ = a² MSE

Raw error metrics therefore should not be compared casually across differently scaled targets.


In [ ]:
y=np.array([100.,200.,300.,400.])
pred=np.array([110.,190.,330.,380.])
def metrics(y,p):
    r=y-p
    return np.mean(np.abs(r)),np.mean(r**2),np.sqrt(np.mean(r**2))
print("Original:",metrics(y,pred))
print("Scaled  :",metrics(y/1000,pred/1000))


## 15. Percentage Error

Absolute Percentage Error:

### APEᵢ = |(𝑦ᵢ − ŷᵢ) / 𝑦ᵢ| × 100%

Mean Absolute Percentage Error:

### MAPE = (100%/n) Σᵢ₌₁ⁿ |(𝑦ᵢ − ŷᵢ) / 𝑦ᵢ|

MAPE is undefined when an actual target is zero and can behave poorly near zero. Intuitive-looking metrics can still have mathematical limitations.


## 16. Residual Distribution Matters

One average compresses an entire distribution of errors into a single number.

Two models can have similar average error while having very different tails and failure patterns.


In [ ]:
rng=np.random.default_rng(11)
A=rng.normal(0,3,3000)
B=np.where(rng.random(3000)<.95,rng.normal(0,1.7,3000),rng.normal(0,11,3000))
for name,r in [("Model A",A),("Model B",B)]:
    print(name,
          "MAE=",round(np.mean(np.abs(r)),3),
          "RMSE=",round(np.sqrt(np.mean(r**2)),3),
          "95th abs error=",round(np.quantile(np.abs(r),.95),3))

plt.figure(figsize=(8,5))
plt.hist(A,bins=60,alpha=.6,label="Model A")
plt.hist(B,bins=60,alpha=.6,label="Model B")
plt.xlabel("Residual"); plt.ylabel("Frequency")
plt.title("Different Error Distributions")
plt.legend(); plt.grid(alpha=.2); plt.show()


## 17. Tail Error

For absolute residuals |e|, a q-th quantile Qq can be described by:

### P(|e| ≤ Qq) ≥ q

If rare large errors are consequential, tail metrics may be more decision-relevant than an average alone.


In [ ]:
for q in [.50,.90,.95,.99]:
    print(f"{q:.0%} absolute-error quantile = {np.quantile(np.abs(B),q):.3f}")


## 18. Comparing Models

There is no meaningful concept of a universally “best” model independent of the evaluation objective.

### Best Model is conditional on the Validation Objective

A ranking based on MAE may differ from a ranking based on RMSE, maximum error, tail error, or application-specific cost.


In [ ]:
y=np.array([100.,110.,120.,130.,140.,150.,160.,170.])
pred_A=np.array([98.,112.,119.,132.,138.,152.,158.,190.])
pred_B=np.array([95.,106.,125.,126.,145.,146.,165.,166.])
for name,p in [("Model A",pred_A),("Model B",pred_B)]:
    r=y-p
    print(name,
          "MAE=",round(np.mean(np.abs(r)),3),
          "RMSE=",round(np.sqrt(np.mean(r**2)),3),
          "MaxAbs=",round(np.max(np.abs(r)),3))


## 19. Error Metrics Have Sampling Uncertainty

An observed MAE or RMSE is estimated from a finite evaluation sample.

### Estimated Performance ≠ True Population Performance

A different sample can produce a different metric value. Validation therefore needs uncertainty analysis around the metric itself.


## 20. Experiment — Bootstrap Uncertainty for MAE

The bootstrap resamples observed evaluation cases with replacement and recalculates the metric many times.


In [ ]:
rng=np.random.default_rng(2026)
y=np.linspace(20,120,80)
pred=y+rng.normal(0,8,len(y))
errors=np.abs(y-pred)
observed=errors.mean()

B=5000
boot=np.empty(B)
for b in range(B):
    boot[b]=rng.choice(errors,size=len(errors),replace=True).mean()

lo,hi=np.quantile(boot,[.025,.975])
print(f"Observed MAE = {observed:.3f}")
print(f"Bootstrap 95% percentile interval = [{lo:.3f}, {hi:.3f}]")

plt.figure(figsize=(8,5))
plt.hist(boot,bins=50)
plt.axvline(observed,linestyle="--",label="Observed MAE")
plt.xlabel("Bootstrap MAE"); plt.ylabel("Frequency")
plt.title("Sampling Uncertainty of MAE")
plt.legend(); plt.grid(alpha=.2); plt.show()


The bootstrap interval is an estimate conditional on the observed sample and resampling procedure. It is not mathematical proof of future performance.


## 21. Baselines

A sophisticated model should often be compared with a meaningful simple reference.

For example:

### ŷbaseline = mean of the training targets

If a complex model cannot improve materially upon a sensible baseline, its practical value may be limited.


In [ ]:
rng=np.random.default_rng(31)
x=np.linspace(0,10,100)
y=4*x+10+rng.normal(0,5,100)
split=70
xt,xe=x[:split],x[split:]
yt,ye=y[:split],y[split:]
w,b=np.polyfit(xt,yt,1)
model=w*xe+b
baseline=np.full_like(ye,yt.mean())
for name,p in [("Linear model",model),("Mean baseline",baseline)]:
    print(name,
          "MAE=",round(np.mean(np.abs(ye-p)),3),
          "RMSE=",round(np.sqrt(np.mean((ye-p)**2)),3))


## 22. From Error to Evidence

### Residuals → Metric → Distribution → Uncertainty → Context → Evidence

A metric is one stage of the scientific argument. It is not the conclusion by itself.

Interpretation requires target units, baseline performance, intended use, consequences of error, subgroup behaviour, tail behaviour, evaluation sample, uncertainty, and deployment conditions.


## 23. OpenVals Perspective

A validation system should not simply report one error score. Depending on the problem, it may need to preserve and evaluate MAE, MSE, RMSE, signed bias, maximum error, error quantiles, residual distributions, subgroup errors, baseline comparisons, confidence intervals, drift in error over time, and cost-sensitive error.

The objective is to move from **metric reporting** toward **evidence-based model validation**.


## 24. Mathematical Summary

### 𝑒ᵢ = 𝑦ᵢ − ŷᵢ

### MAE = (1/n) Σᵢ₌₁ⁿ |𝑦ᵢ − ŷᵢ|

### MSE = (1/n) Σᵢ₌₁ⁿ (𝑦ᵢ − ŷᵢ)²

### RMSE = √[(1/n) Σᵢ₌₁ⁿ (𝑦ᵢ − ŷᵢ)²]

### MAE ≤ RMSE

### c*MSE = E[Y]

### c*MAE = Median(Y)


## 25. Central Conclusion

### Error Metric ≠ Complete Evidence

> **How we measure error changes what we conclude about a model.**

MAE asks about average absolute deviation. MSE strongly penalizes large residuals. RMSE retains that sensitivity while returning to the original target units.

None is universally superior. The scientifically appropriate metric depends on the error distribution, decision consequences, target scale, intended use, and uncertainty of the estimate.

### Next Episode

**Episode 05 — Generalization: Why a Model That Performs Well Today Can Fail Tomorrow**

The next episode examines training error, test error, overfitting, underfitting, bias–variance trade-off, cross-validation, and the mathematics of generalization.
